# Sumarização de Textos com PLN: Algoritmo Baseado em Frequência
**Autor: Wellington M Santos - Data Scientist**  
[![LinkedIn](https://img.shields.io/badge/LinkedIn-wellington--moreira--santos-blue)](https://www.linkedin.com/in/wellington-moreira-santos)
[![Email](https://img.shields.io/badge/Email-wsantos08%40hotmail.com-red)](mailto:wsantos08@hotmail.com)



## 1. Introdução

Toda semana leio mais artigos do que consigo processar. Newsletters, papers, posts técnicos, portais de notícias: o volume de texto disponível cresce mais rápido do que minha capacidade de leitura. Esse é exatamente o problema que a sumarização automática de textos se propõe a resolver.

Neste projeto, implemento do zero um algoritmo de sumarização extrativa baseado em frequência de palavras. A ideia central é simples: palavras que aparecem com mais frequência em um texto carregam mais informação sobre o seu tema; sentenças que concentram essas palavras merecem estar no resumo. Sem redes neurais, sem embeddings, sem modelos pesados. Só estatística básica e PLN clássico.

O caminho que percorro aqui é progressivo. Começo com um texto curto construído manualmente, onde cada etapa do algoritmo fica completamente visível. Depois encapsulo tudo em funções reutilizáveis e aplico sobre artigos reais extraídos da web. Adiciono uma variante com lematização via spaCy para comparar o efeito do pré-processamento sobre o resultado. E fecho com uma avaliação quantitativa usando ROUGE, a métrica padrão para sumarização.

**Stack utilizada:** `Python 3.10+`, `nltk`, `collections`, `heapq`, `re`, `string`, `newspaper4k`, `spacy>=3.x`, `rouge-score`, `IPython.display`

**Seções:**

1. Introdução
2. Instalação e Configuração
3. Pré-processamento do Texto
4. Frequência das Palavras
5. Tokenização de Sentenças
6. Geração do Resumo
7. Visualização do Resumo
8. Extração de Texto da Web
9. Funções Reutilizáveis
10. Sumarização em Lote
11. Extensão: Lematização com spaCy
12. Avaliação com ROUGE
13. Conclusão e Considerações Finais


## 2. Instalação e Configuração


In [1]:
# dependências :: executar uma vez
# !pip install nltk newspaper4k spacy rouge-score
# !python -m spacy download pt_core_news_sm

import re
import string
import heapq
from collections import Counter
from pprint import pprint

import nltk
from IPython.display import HTML, display

nltk.download('punkt')
nltk.download('punkt_tab')
nltk.download('stopwords')

print('- BIBLIOTECAS CARREGADAS -')

- BIBLIOTECAS CARREGADAS -


[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\wsant\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\wsant\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\wsant\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!



## 3. Pré-processamento do Texto

Antes de contar qualquer coisa, preciso normalizar o texto. O objetivo é eliminar o ruído linguístico e ficar só com os tokens que carregam significado temático: sem maiúsculas, sem pontuação, sem stopwords.

Stopwords são palavras funcionais de alta frequência que aparecem em qualquer texto independentemente do assunto, como "de", "que", "para", "é". Se eu as mantivesse, elas inflariam artificialmente a pontuação de todas as sentenças de forma indistinta, o que é exatamente o que não quero.

O texto de exemplo abaixo tem repetição propositalmente elevada da palavra "inteligência". Faço isso para tornar o comportamento do algoritmo visível já nessa etapa inicial.


In [2]:
texto_original = """A inteligência artificial é a inteligência similar à humana.
                    Definem como o estudo de agente artificial com inteligência.
                    Ciência e engenharia de produzir máquinas com inteligência.
                    Resolver problemas e possuir inteligência.
                    Relacionada ao comportamento inteligente.
                    Construção de máquinas para raciocinar.
                    Aprender com os erros e acertos.
                    Inteligência artificial é raciocinar nas situações do cotidiano."""


# Remove espaços múltiplos e quebras de linha
texto_original = re.sub(r'\s+', ' ', texto_original)
pprint(texto_original)

('A inteligência artificial é a inteligência similar à humana. Definem como o '
 'estudo de agente artificial com inteligência. Ciência e engenharia de '
 'produzir máquinas com inteligência. Resolver problemas e possuir '
 'inteligência. Relacionada ao comportamento inteligente. Construção de '
 'máquinas para raciocinar. Aprender com os erros e acertos. Inteligência '
 'artificial é raciocinar nas situações do cotidiano.')


In [3]:
stopwords = nltk.corpus.stopwords.words('portuguese')
print(f"Total de stopwords em português: {len(stopwords)}")

Total de stopwords em português: 207


In [5]:
def preprocessamento(texto):
    """
    Normaliza o texto: lowercase, tokenização, remoção de stopwords e pontuação.

    Parâmetros:
        texto (str): texto bruto de entrada

    Retorna:
        str: string com tokens relevantes separados por espaço
    """
    texto_formatado = texto.lower()
    tokens = nltk.word_tokenize(texto_formatado, language='portuguese')
    tokens = [
        palavra for palavra in tokens
        if palavra not in stopwords and palavra not in string.punctuation
    ]
    texto_formatado = ' '.join([t for t in tokens if not t.isdigit()])
    return texto_formatado


texto_formatado = preprocessamento(texto_original)
pprint(texto_formatado)

('inteligência artificial inteligência similar humana definem estudo agente '
 'artificial inteligência ciência engenharia produzir máquinas inteligência '
 'resolver problemas possuir inteligência relacionada comportamento '
 'inteligente construção máquinas raciocinar aprender erros acertos '
 'inteligência artificial raciocinar situações cotidiano')



O texto resultante é consideravelmente menor. A palavra "inteligência" sobreviveu em quase todas as posições e vai ter peso direto na próxima etapa.



## 4. Frequência das Palavras

Com o texto normalizado, conto a frequência de cada token usando `Counter`. Em seguida, normalizo os valores dividindo pela frequência máxima observada. O resultado são pesos proporcionais entre 0 e 1, o que permite comparar textos de tamanhos diferentes sem que o comprimento distorça os resultados.


In [6]:
contagem = Counter(nltk.word_tokenize(texto_formatado))
print(f"Vocabulário: {len(contagem)} tokens únicos")
print(contagem.most_common(10))

Vocabulário: 24 tokens únicos
[('inteligência', 6), ('artificial', 3), ('máquinas', 2), ('raciocinar', 2), ('similar', 1), ('humana', 1), ('definem', 1), ('estudo', 1), ('agente', 1), ('ciência', 1)]


In [7]:
frequencia_maxima = max(contagem.values())

frequencia_palavras = {
    palavra: freq / frequencia_maxima
    for palavra, freq in contagem.items()
}

pprint(frequencia_palavras)

{'acertos': 0.16666666666666666,
 'agente': 0.16666666666666666,
 'aprender': 0.16666666666666666,
 'artificial': 0.5,
 'ciência': 0.16666666666666666,
 'comportamento': 0.16666666666666666,
 'construção': 0.16666666666666666,
 'cotidiano': 0.16666666666666666,
 'definem': 0.16666666666666666,
 'engenharia': 0.16666666666666666,
 'erros': 0.16666666666666666,
 'estudo': 0.16666666666666666,
 'humana': 0.16666666666666666,
 'inteligente': 0.16666666666666666,
 'inteligência': 1.0,
 'máquinas': 0.3333333333333333,
 'possuir': 0.16666666666666666,
 'problemas': 0.16666666666666666,
 'produzir': 0.16666666666666666,
 'raciocinar': 0.3333333333333333,
 'relacionada': 0.16666666666666666,
 'resolver': 0.16666666666666666,
 'similar': 0.16666666666666666,
 'situações': 0.16666666666666666}


Após a normalização, a palavra mais frequente recebe peso 1.0. No texto de exemplo, "inteligência" domina o vocabulário e vai guiar a seleção das sentenças na etapa de pontuação.


## 5. Tokenização de Sentenças

A sumarização extrativa funciona selecionando sentenças completas do texto original. Para isso, preciso separar o texto em sentenças de forma confiável. Dividir por ponto final simples falha em casos como "Dr.", "Prof." ou siglas. O NLTK resolve isso com `sent_tokenize`, que usa um modelo pré-treinado para detectar fronteiras reais de sentença.


In [8]:
# Demonstração do problema com split simples
exemplo = 'o dr. joão foi para casa. Ele chegou cedo.'
print("split por ponto:", exemplo.split('.'))
print("sent_tokenize: ", nltk.sent_tokenize(exemplo, language='portuguese'))

split por ponto: ['o dr', ' joão foi para casa', ' Ele chegou cedo', '']
sent_tokenize:  ['o dr. joão foi para casa.', 'Ele chegou cedo.']



A diferença é imediata. O `split('.')` fragmenta "dr." em dois pedaços inválidos. O `sent_tokenize` reconhece a abreviação e mantém a sentença intacta.


In [9]:
lista_sentencas = nltk.sent_tokenize(texto_original, language='portuguese')
print(f"Total de sentenças: {len(lista_sentencas)}")
for i, s in enumerate(lista_sentencas, 1):
    print(f"  [{i}] {s}")

Total de sentenças: 8
  [1] A inteligência artificial é a inteligência similar à humana.
  [2] Definem como o estudo de agente artificial com inteligência.
  [3] Ciência e engenharia de produzir máquinas com inteligência.
  [4] Resolver problemas e possuir inteligência.
  [5] Relacionada ao comportamento inteligente.
  [6] Construção de máquinas para raciocinar.
  [7] Aprender com os erros e acertos.
  [8] Inteligência artificial é raciocinar nas situações do cotidiano.



## 6. Geração do Resumo

Com as frequências normalizadas e as sentenças identificadas, posso pontuar cada sentença. Para cada uma, itero sobre seus tokens e acumulo os pesos das palavras que constam no dicionário de frequências. A sentença que acumular mais peso é, segundo esse critério, a mais representativa do texto.


In [10]:
nota_sentencas = {}

for sentenca in lista_sentencas:
    tokens = nltk.word_tokenize(sentenca.lower(), language='portuguese')
    for palavra in tokens:
        if palavra in frequencia_palavras:
            nota_sentencas[sentenca] = (
                nota_sentencas.get(sentenca, 0) + frequencia_palavras[palavra]
            )

pprint(nota_sentencas)

{'A inteligência artificial é a inteligência similar à humana.': 2.833333333333333,
 'Aprender com os erros e acertos.': 0.5,
 'Ciência e engenharia de produzir máquinas com inteligência.': 1.8333333333333333,
 'Construção de máquinas para raciocinar.': 0.8333333333333333,
 'Definem como o estudo de agente artificial com inteligência.': 2.0,
 'Inteligência artificial é raciocinar nas situações do cotidiano.': 2.1666666666666665,
 'Relacionada ao comportamento inteligente.': 0.5,
 'Resolver problemas e possuir inteligência.': 1.5}


In [11]:
# Seleciona as 3 sentenças com maior pontuação
melhores_sentencas = heapq.nlargest(3, nota_sentencas, key=nota_sentencas.get)

resumo = ' '.join(melhores_sentencas)
print("Resumo gerado:")
pprint(resumo)

Resumo gerado:
('A inteligência artificial é a inteligência similar à humana. Inteligência '
 'artificial é raciocinar nas situações do cotidiano. Definem como o estudo de '
 'agente artificial com inteligência.')



Uso `heapq.nlargest` em vez de ordenar o dicionário inteiro. É mais eficiente quando só me interessa o topo da lista. O parâmetro `n` é um hiperparâmetro configurável e deve ser ajustado proporcionalmente ao tamanho do texto original.

## 7. Visualização do Resumo

Uma forma prática de avaliar a qualidade do resumo é destacar, no texto original, as sentenças selecionadas. Consigo ver de imediato se o algoritmo capturou trechos realmente relevantes ou se foi guiado pela repetição mecânica de palavras.

A função abaixo detecta automaticamente o ambiente de execução. Em Jupyter exibe HTML com destaque visual; em outros ambientes imprime em texto puro.


In [12]:
def visualiza_resumo(titulo, lista_sentencas, melhores_sentencas):
    """
    Exibe o texto original com as sentenças do resumo destacadas.
    Suporta Jupyter (HTML) e outros ambientes (texto puro).

    Parâmetros:
        titulo (str): título exibido no cabeçalho
        lista_sentencas (list): todas as sentenças do texto original
        melhores_sentencas (list): sentenças selecionadas para o resumo
    """
    try:
        get_ipython  # noqa
        # Ambiente Jupyter: exibe HTML com destaque
        texto_html = ''
        for sentenca in lista_sentencas:
            if sentenca in melhores_sentencas:
                texto_html += f'<mark>{sentenca}</mark> '
            else:
                texto_html += sentenca + ' '
        display(HTML(f'<h3>Resumo: {titulo}</h3><p>{texto_html}</p>'))
    except NameError:
        # Fallback para terminais e scripts
        print(f"\n=== Resumo: {titulo} ===")
        for sentenca in lista_sentencas:
            marcador = ">> " if sentenca in melhores_sentencas else "   "
            print(f"{marcador}{sentenca}")


visualiza_resumo('Texto de Exemplo', lista_sentencas, melhores_sentencas)


## 8. Extração de Texto da Web

Textos construídos manualmente são úteis para entender o algoritmo, mas o caso de uso real é aplicar a sumarização sobre artigos publicados na internet. Para isso, uso o `newspaper4k`, fork ativo do antigo `newspaper3k` com suporte a Python 3.10+ e melhor extração de conteúdo em português.


In [14]:
# !pip install newspaper4k
from newspaper import Article

In [15]:
def extrair_artigo(url):
    """
    Extrai título e texto principal de uma URL usando newspaper4k.

    Parâmetros:
        url (str): endereço do artigo

    Retorna:
        tuple: (titulo, texto) ou (None, None) em caso de erro
    """
    try:
        artigo = Article(url, language='pt')
        artigo.download()
        artigo.parse()
        return artigo.title, artigo.text
    except Exception as e:
        print(f"Erro ao extrair {url}: {e}")
        return None, None

In [17]:
url = 'https://agenciabrasil.ebc.com.br/economia/noticia/2024-01/fmi-inteligencia-artificial-afetara-40-dos-empregos-em-todo-o-mundo'
titulo, texto = extrair_artigo(url)

print(f"Título: {titulo}")
print(f"Tamanho do texto: {len(texto)} caracteres")
print(f"\nPrimeiros 300 caracteres:\n{texto[:300]}...")

Título: FMI: inteligência artificial afetará 40% dos empregos em todo o mundo
Tamanho do texto: 2124 caracteres

Primeiros 300 caracteres:
O desenvolvimento da inteligência artificial (IA) terá consequências para 40% dos empregos em todo o mundo, sobretudo nas economias avançadas, disse a diretora-geral do Fundo Monetário Internacional (FMI).

"No mundo, 40% dos empregos serão afetados. E mais: será o caso de quanto mais qualificado fo...


In [18]:
artigo_original = texto
artigo_formatado = preprocessamento(artigo_original)
print(f"Tamanho após pré-processamento: {len(artigo_formatado)} caracteres")
print(f"Redução: {100 - len(artigo_formatado) * 100 // len(artigo_original)}%")

Tamanho após pré-processamento: 1634 caracteres
Redução: 24%


Em artigos jornalísticos em português, a redução pelo pré-processamento costuma ficar entre 40% e 60%, dependendo da densidade de stopwords e pontuação no texto original.

## 9. Funções Reutilizáveis

Com o pipeline funcionando de ponta a ponta, encapsulo toda a lógica na função `sumarizar`. Ela recebe qualquer texto e um número de sentenças desejado, e devolve os objetos intermediários para quem quiser inspecionar o processo.


In [19]:
def sumarizar(texto, quantidade_sentencas):
    """
    Executa o pipeline completo de sumarização por frequência.

    Parâmetros:
        texto (str): texto original a ser sumarizado
        quantidade_sentencas (int): número de sentenças no resumo

    Retorna:
        tuple: (lista_sentencas, melhores_sentencas, frequencia_palavras, nota_sentencas)
    """
    texto_formatado = preprocessamento(texto)

    contagem = Counter(nltk.word_tokenize(texto_formatado))
    frequencia_maxima = max(contagem.values())
    frequencia_palavras = {p: f / frequencia_maxima for p, f in contagem.items()}

    lista_sentencas = nltk.sent_tokenize(texto, language='portuguese')

    nota_sentencas = {}
    for sentenca in lista_sentencas:
        tokens = nltk.word_tokenize(sentenca.lower(), language='portuguese')
        for palavra in tokens:
            if palavra in frequencia_palavras:
                nota_sentencas[sentenca] = (
                    nota_sentencas.get(sentenca, 0) + frequencia_palavras[palavra]
                )

    melhores_sentencas = heapq.nlargest(
        quantidade_sentencas, nota_sentencas, key=nota_sentencas.get
    )

    return lista_sentencas, melhores_sentencas, frequencia_palavras, nota_sentencas

In [20]:

lista_sentencas, melhores_sentencas, frequencia_palavras, nota_sentencas = sumarizar(artigo_original, 5)

print(f"Total de sentenças no artigo: {len(lista_sentencas)}")
print(f"\nSentenças selecionadas para o resumo:")

for i, s in enumerate(melhores_sentencas, 1):
    print(f"  [{i}] {s}")

Total de sentenças no artigo: 15

Sentenças selecionadas para o resumo:
  [1] O documento alerta que a IA poderá agravar as desigualdades salariais, prejudicando sobretudo a classe média, enquanto os trabalhadores com rendimentos já elevados poderão ver os seus salários "aumentarem mais do que a proporção" dos ganhos de produtividade com essa tecnologia.
  [2] "É certo que haverá impacto" disse Georgieva, observando que a IA pode acabar com alguns empregos e melhorar outros.
  [3] "Devemos concentrar-nos nos países de rendimento mais baixo", destacou a diretora-geral do FMI, que demonstrou receio com o risco de abandono escolar nos Estados mais pobres.
  [4] "A IA pode ser assustadora, mas também pode ser uma grande oportunidade para todos", observou.
  [5] A diretora defendeu que a prioridade deve ser ajudar os trabalhadores afetados e "partilhar os ganhos de produtividade".


In [21]:
visualiza_resumo(titulo, lista_sentencas, melhores_sentencas)


## 10. Sumarização em Lote

Com as funções prontas, processar múltiplos artigos é uma iteração sobre uma lista de URLs. Extraio o texto de cada página com `newspaper4k` e passo direto para o pipeline.


In [22]:
lista_urls = [
    'https://agenciabrasil.ebc.com.br/economia/noticia/2024-01/fmi-inteligencia-artificial-afetara-40-dos-empregos-em-todo-o-mundo',
    'https://agenciabrasil.ebc.com.br/geral/noticia/2024-01/inss-testa-inteligencia-artificial-para-identificar-fraudes',
    'https://agenciabrasil.ebc.com.br/geral/noticia/2024-07/brasil-quer-ter-supercomputador-e-desenvolver-modelos-nacionais-de-ia',
]

for url in lista_urls:
    titulo_artigo, texto_artigo = extrair_artigo(url)
    if texto_artigo:
        lista_s, melhores_s, _, _ = sumarizar(texto_artigo, 5)
        visualiza_resumo(titulo_artigo, lista_s, melhores_s)


Os artigos cobrem temas distintos, mas o pipeline se comporta de forma idêntica para todos. Textos com vocabulário mais coeso e menos dispersão temática tendem a gerar resumos mais representativos.



## 11. Extensão: Lematização com spaCy

Na função `preprocessamento` original, os tokens são usados como estão. Isso significa que "corrida", "corrido" e "correr" são tratados como três palavras distintas, cada uma com sua própria contagem de frequência. A lematização resolve isso: ela reduz cada token à sua forma canônica, o lema, fazendo com que variações morfológicas da mesma palavra somem suas frequências.

Uso o spaCy 3.x com o modelo oficial para português, `pt_core_news_sm`. É mais preciso e bem mantido.


In [23]:
# !python -m spacy download pt_core_news_sm
import spacy

pln = spacy.load('pt_core_news_sm')

In [24]:
# Demonstração da lematização
frase = 'inteligentes inteligente inteligência corrida corrido correr correndo'
documento = pln(frase)
for token in documento:
    print(f"{token.text:20} -> {token.lemma_}")

inteligentes         -> inteligente
inteligente          -> inteligente
inteligência         -> inteligência
corrida              -> corrido
corrido              -> corrir
correr               -> correr
correndo             -> correr


In [25]:
def preprocessamento_lematizacao(texto):
    """
    Variante do pré-processamento com lematização via spaCy 3.x.
    Reduz variações morfológicas ao lema antes de calcular frequências.

    Parâmetros:
        texto (str): texto bruto de entrada

    Retorna:
        str: string com lemas relevantes separados por espaço
    """
    texto = re.sub(r'\s+', ' ', texto.lower())
    documento = pln(texto)
    tokens = [
        token.lemma_ for token in documento
        if token.lemma_ not in stopwords
        and token.lemma_ not in string.punctuation
        and not token.is_digit
        and not token.is_space
    ]
    return ' '.join(tokens)


In [26]:
print("Sem lematização:")
print(preprocessamento(texto_original))
print("\nCom lematização:")
print(preprocessamento_lematizacao(texto_original))

Sem lematização:
inteligência artificial inteligência similar humana definem estudo agente artificial inteligência ciência engenharia produzir máquinas inteligência resolver problemas possuir inteligência relacionada comportamento inteligente construção máquinas raciocinar aprender erros acertos inteligência artificial raciocinar situações cotidiano

Com lematização:
inteligência artificial inteligência similar a o humano definir estudo agente artificial inteligência ciência engenharia produzir máquina inteligência resolver problema possuir inteligência relacionar a o comportamento inteligente construção máquina raciocinar aprender erro acerto inteligência artificial raciocinar em o situação de o cotidiano


In [27]:
def sumarizar_lematizacao(texto, quantidade_sentencas):
    """
    Pipeline de sumarização com pré-processamento por lematização.
    Mesma estrutura de sumarizar(), substituindo a função de pré-processamento.
    """
    texto_formatado = preprocessamento_lematizacao(texto)

    contagem = Counter(nltk.word_tokenize(texto_formatado))
    frequencia_maxima = max(contagem.values())
    frequencia_palavras = {p: f / frequencia_maxima for p, f in contagem.items()}

    lista_sentencas = nltk.sent_tokenize(texto, language='portuguese')

    nota_sentencas = {}
    for sentenca in lista_sentencas:
        tokens = nltk.word_tokenize(sentenca.lower(), language='portuguese')
        for palavra in tokens:
            if palavra in frequencia_palavras:
                nota_sentencas[sentenca] = (
                    nota_sentencas.get(sentenca, 0) + frequencia_palavras[palavra]
                )

    melhores_sentencas = heapq.nlargest(
        quantidade_sentencas, nota_sentencas, key=nota_sentencas.get
    )
    return lista_sentencas, melhores_sentencas, frequencia_palavras, nota_sentencas

In [28]:
for url in lista_urls:
    titulo_artigo, texto_artigo = extrair_artigo(url)
    if texto_artigo:
        lista_s, melhores_s, _, _ = sumarizar_lematizacao(texto_artigo, 5)
        visualiza_resumo(f"{titulo_artigo} (lematização)", lista_s, melhores_s)

## 12. Avaliação com ROUGE

Até aqui avaliei os resultados de forma qualitativa, lendo os resumos gerados. Para fechar o projeto com rigor, adiciono uma avaliação quantitativa usando ROUGE (Recall-Oriented Understudy for Gisting Evaluation), a métrica padrão para sumarização automática.

ROUGE compara o resumo gerado pelo algoritmo com um resumo de referência escrito por um humano. As métricas principais são:

- **ROUGE-1:** sobreposição de unigramas (palavras individuais)
- **ROUGE-2:** sobreposição de bigramas (pares de palavras consecutivas)
- **ROUGE-L:** maior subsequência comum entre os dois textos

Scores mais altos indicam maior similaridade com o resumo de referência. Para fins didáticos, uso um resumo de referência escrito manualmente a partir do artigo extraído.


In [29]:
# !pip install rouge-score
from rouge_score import rouge_scorer

In [34]:
# Resumo de referência escrito manualmente a partir do artigo do FMI
# Fonte: https://agenciabrasil.ebc.com.br/economia/noticia/2024-01/fmi-inteligencia-artificial-afetara-40-dos-empregos-em-todo-o-mundo
resumo_referencia = """
O FMI alerta que a inteligência artificial afetará 40% dos empregos em todo o mundo,
com impacto ainda maior nas economias avançadas, onde 60% dos postos de trabalho
serão impactados. A diretora-geral do fundo ressalta que os efeitos não são
necessariamente negativos e podem resultar em aumento de rendimentos, mas alerta
para o risco de aprofundamento das desigualdades entre países com diferentes
capacidades de adaptação tecnológica.
"""

# Resumo gerado pelo algoritmo
lista_s, melhores_s, _, _ = sumarizar(artigo_original, 5)
resumo_gerado = ' '.join(melhores_s)

In [31]:
def avaliar_rouge(resumo_gerado, resumo_referencia):
    """
    Calcula métricas ROUGE entre o resumo gerado e o de referência.

    Parâmetros:
        resumo_gerado (str): resumo produzido pelo algoritmo
        resumo_referencia (str): resumo de referência escrito por humano

    Retorna:
        dict: scores ROUGE-1, ROUGE-2 e ROUGE-L (precision, recall, f1)
    """
    scorer = rouge_scorer.RougeScorer(['rouge1', 'rouge2', 'rougeL'], use_stemmer=False)
    scores = scorer.score(resumo_referencia.strip(), resumo_gerado.strip())
    return scores

In [35]:
scores = avaliar_rouge(resumo_gerado, resumo_referencia)

print("Avaliação ROUGE")
print("-" * 45)
for metrica, resultado in scores.items():
    print(f"{metrica.upper():10} | "
          f"Precision: {resultado.precision:.3f} | "
          f"Recall: {resultado.recall:.3f} | "
          f"F1: {resultado.fmeasure:.3f}")

Avaliação ROUGE
---------------------------------------------
ROUGE1     | Precision: 0.254 | Recall: 0.427 | F1: 0.318
ROUGE2     | Precision: 0.064 | Recall: 0.108 | F1: 0.080
ROUGEL     | Precision: 0.143 | Recall: 0.240 | F1: 0.179


In [36]:
# Comparação entre as duas abordagens de pré-processamento
lista_s_lem, melhores_s_lem, _, _ = sumarizar_lematizacao(artigo_original, 5)
resumo_lematizacao = ' '.join(melhores_s_lem)

scores_base = avaliar_rouge(resumo_gerado, resumo_referencia)
scores_lem  = avaliar_rouge(resumo_lematizacao, resumo_referencia)

print("Comparação: Tokenização vs. Lematização")
print("-" * 50)
print(f"{'Métrica':10} {'Tokenização F1':>18} {'Lematização F1':>18}")
print("-" * 50)
for metrica in ['rouge1', 'rouge2', 'rougeL']:
    f1_base = scores_base[metrica].fmeasure
    f1_lem  = scores_lem[metrica].fmeasure
    melhor  = "<-- melhor" if f1_lem > f1_base else ""
    print(f"{metrica.upper():10} {f1_base:>18.3f} {f1_lem:>18.3f} {melhor}")

Comparação: Tokenização vs. Lematização
--------------------------------------------------
Métrica        Tokenização F1     Lematização F1
--------------------------------------------------
ROUGE1                  0.318              0.435 <-- melhor
ROUGE2                  0.080              0.196 <-- melhor
ROUGEL                  0.179              0.269 <-- melhor



Os scores ROUGE dependem diretamente da qualidade do resumo de referência. Para um projeto de estudo, o mais importante é observar a diferença relativa entre as duas abordagens, não os valores absolutos.



## 13. Conclusão

### 13.1 Síntese do Projeto

Neste projeto implementei um pipeline completo de sumarização extrativa baseada em frequência de palavras. O algoritmo percorre quatro etapas: pré-processamento do texto, cálculo da frequência proporcional dos tokens, pontuação das sentenças pela soma dos pesos das palavras que contêm, e seleção das melhores via `heapq.nlargest`.

O pipeline foi construído de forma incremental: comecei com um texto de exemplo para entender a lógica, depois extraí artigos reais da web com `newspaper4k`, encapsulei tudo em funções reutilizáveis para processamento em lote, comparei duas estratégias de pré-processamento (tokenização simples e lematização via spaCy 3.x) e avaliei os resultados quantitativamente com ROUGE.

### 13.2 Limitações

O algoritmo é sensível à repetição mecânica de palavras. Um texto que repete um termo muitas vezes favorece sentenças que o contêm, mesmo que não sejam as mais informativas. Como o método é puramente extrativo, o resumo pode ter problemas de coesão: sentenças retiradas do contexto original às vezes soam desconexas quando lidas em sequência.

A avaliação com ROUGE também tem limitações: ela mede sobreposição léxica, não compreensão semântica. Um resumo semanticamente preciso mas com vocabulário diferente do de referência pode receber scores baixos.

> Material de estudo desenvolvido para o repositório [data-trivium](https://github.com/esscova/data-trivium).  